# Clasificación de triatominos con Vision Transformer

Fine-tuning de `google/vit-base-patch16-224` sobre el dataset `Totan2305/triatominos-augmentado`
(30.000 imágenes, 3 géneros: Panstrongylus, Rhodnius, Triatoma).

Requiere GPU:
- Colab: Entorno de ejecución → Cambiar tipo de entorno → GPU T4.
- Kaggle: Settings → Accelerator → GPU; Internet → On.

## Dependencias

In [ ]:
# scikit-learn y matplotlib ya vienen instalados en Colab y Kaggle.
!pip install -q -U transformers datasets accelerate

## Entorno

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device)
if device == "cuda":
    print(torch.cuda.get_device_name(0))

## Datos

El dataset está en formato Parquet (repositorio público), por lo que se carga en
segundos. Las etiquetas se derivan de las carpetas por clase.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("Totan2305/triatominos-augmentado-parquet")
data = dataset["train"]

labels = data.features["label"].names
num_labels = len(labels)
id2label = {i: c for i, c in enumerate(labels)}
label2id = {c: i for i, c in enumerate(labels)}

print(data.num_rows, "imágenes -", labels)

## Partición train / validación

85 / 15 estratificado.

In [ ]:
split = data.train_test_split(test_size=0.15, stratify_by_column="label", seed=42)
train_ds, val_ds = split["train"], split["test"]
print("train:", train_ds.num_rows, "- val:", val_ds.num_rows)

## Preprocesamiento

In [ ]:
from transformers import AutoImageProcessor
from torchvision.transforms import Compose, Resize, ToTensor, Normalize

checkpoint = "google/vit-base-patch16-224"
processor = AutoImageProcessor.from_pretrained(checkpoint)

size = processor.size.get("height", processor.size.get("shortest_edge", 224))
transform = Compose([
    Resize((size, size)),
    ToTensor(),
    Normalize(mean=processor.image_mean, std=processor.image_std),
])

def apply_transform(batch):
    batch["pixel_values"] = [transform(img.convert("RGB")) for img in batch["image"]]
    return batch

train_ds.set_transform(apply_transform)
val_ds.set_transform(apply_transform)

## Modelo

Se sustituye el cabezal de clasificación de ImageNet por uno de 3 clases.

In [ ]:
from transformers import AutoModelForImageClassification

model = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

## Entrenamiento

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def collate_fn(batch):
    pixel_values = torch.stack([x["pixel_values"] for x in batch])
    labels = torch.tensor([x["label"] for x in batch])
    return {"pixel_values": pixel_values, "labels": labels}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="vit-triatominos",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    remove_unused_columns=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

In [ ]:
trainer.train()

## Evaluación

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

pred = trainer.predict(val_ds)
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=1)

print(classification_report(y_true, y_pred, target_names=labels, digits=4))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay(cm, display_labels=labels).plot(cmap="Blues", ax=ax, xticks_rotation=45)
plt.tight_layout()
plt.savefig("matriz_confusion.png", dpi=150)
plt.show()

## Exportación del modelo

Guarda el modelo y lo publica en Hugging Face (queda permanente y descargable
cuando se quiera). El token se toma de los secretos de Kaggle o Colab.

In [ ]:
output = "modelo_triatominos_vit"
trainer.save_model(output)
processor.save_pretrained(output)


def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None


token = hf_token()
if token:
    model.push_to_hub("Totan2305/triatominos-vit", token=token)
    processor.push_to_hub("Totan2305/triatominos-vit", token=token)
    print("Modelo publicado: https://huggingface.co/Totan2305/triatominos-vit")
else:
    print("Sin token de HF; el modelo quedó guardado localmente en:", output)